# 2. Feature Engineering and EDA

**Goal:** turn the cleaned purchases into one row per customer per month, with features that describe the customer's **past 365 days** and a target that describes the **next 90 days**:

- **Churned = 1:** the customer makes no purchase in the next 90 days.
- **Churned = 0:** the customer buys again.

We first build the features step by step for one date (1 December 2010), then repeat them for every month, and finally explore the data.

# Read Data

Load the cleaned purchases. Keep the IDs as text and InvoiceDate as a date.

In [1]:
import pandas as pd

df = pd.read_csv(
    "data/cleaned_purchases.csv.gz",
    dtype={"Invoice": "string", "StockCode": "string", "Customer ID": "string"},
    parse_dates=["InvoiceDate"]
)

df.shape

(775294, 8)

In [2]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


# Feature Engineering

TransactionValue is Quantity multiplied by Price for each purchase row.

In [3]:
df["TransactionValue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "TransactionValue"]].head()

,Quantity,Price,TransactionValue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


### Purchase history

Use the previous 365 days, ending before the prediction date. All feature calculations below use this history.

In [4]:
prediction_date = pd.Timestamp("2010-12-01")
history_start = prediction_date - pd.Timedelta(days=365)

history = df[
    (df["InvoiceDate"] >= history_start)
    & (df["InvoiceDate"] < prediction_date)
].copy()

history["InvoiceDate"].agg(["min", "max"])

min   2009-12-01 07:45:00
max   2010-11-30 19:35:00
Name: InvoiceDate, dtype: datetime64[us]

### Frequency and monetary value

Count each customer's invoices, total their purchase amounts and count their distinct products.

In [5]:
customer_features = history.groupby("Customer ID").agg(
    InvoiceCountLast365Days=("Invoice", "nunique"),
    PurchaseAmountLast365Days=("TransactionValue", "sum"),
    DistinctProductCountLast365Days=("StockCode", "nunique")
).reset_index()

customer_features.head()


,Customer ID,InvoiceCountLast365Days,PurchaseAmountLast365Days,DistinctProductCountLast365Days
0,12346,2,169.36,24
1,12347,1,611.53,40
2,12348,1,221.16,19
3,12349,2,2221.14,89
4,12351,1,300.93,21


Average order value = purchase amount divided by invoice count.

In [6]:
customer_features["AverageOrderValueLast365Days"] = (
    customer_features["PurchaseAmountLast365Days"]
    / customer_features["InvoiceCountLast365Days"]
)

customer_features.head()


,Customer ID,InvoiceCountLast365Days,PurchaseAmountLast365Days,DistinctProductCountLast365Days,AverageOrderValueLast365Days
0,12346,2,169.36,24,84.68
1,12347,1,611.53,40,611.53
2,12348,1,221.16,19,221.16
3,12349,2,2221.14,89,1110.57
4,12351,1,300.93,21,300.93


### Recency and tenure

Days since the customer's last purchase and since their first purchase in the history window.

In [7]:
customer_dates = history.groupby("Customer ID").agg(
    FirstPurchaseDate=("InvoiceDate", "min"),
    LastPurchaseDate=("InvoiceDate", "max")
).reset_index()

customer_dates["DaysSinceLastPurchase"] = (
    prediction_date - customer_dates["LastPurchaseDate"].dt.normalize()
).dt.days

customer_dates["DaysSinceFirstPurchase"] = (
    prediction_date - customer_dates["FirstPurchaseDate"].dt.normalize()
).dt.days

customer_dates.head()


,Customer ID,FirstPurchaseDate,LastPurchaseDate,DaysSinceLastPurchase,DaysSinceFirstPurchase
0,12346,2010-03-02 13:08:00,2010-06-28 13:53:00,156,274
1,12347,2010-10-31 14:20:00,2010-10-31 14:20:00,31,31
2,12348,2010-09-27 14:59:00,2010-09-27 14:59:00,65,65
3,12349,2010-04-29 13:20:00,2010-10-28 08:23:00,34,216
4,12351,2010-11-29 15:23:00,2010-11-29 15:23:00,2,2


### Purchase rhythm

Average gap between the customer's purchase days. One purchase day gives no gap, so the result stays missing.

In [8]:
history["PurchaseDay"] = history["InvoiceDate"].dt.normalize()

purchase_days = (
    history.groupby("Customer ID")["PurchaseDay"]
    .nunique()
    .reset_index(name="PurchaseDayCount")
)

# Build a new table so re-running this cell does not duplicate columns.
customer_rhythm = customer_dates.merge(purchase_days, on="Customer ID", how="left")

span_days = (
    customer_rhythm["LastPurchaseDate"].dt.normalize()
    - customer_rhythm["FirstPurchaseDate"].dt.normalize()
).dt.days

gap_count = (customer_rhythm["PurchaseDayCount"] - 1).replace(0, float("nan"))

customer_rhythm["AverageDaysBetweenPurchases"] = span_days / gap_count

customer_rhythm[["Customer ID", "PurchaseDayCount", "AverageDaysBetweenPurchases"]].head()


,Customer ID,PurchaseDayCount,AverageDaysBetweenPurchases
0,12346,2,118.0
1,12347,1,NaN
2,12348,1,NaN
3,12349,2,182.0
4,12351,1,NaN


OverdueRatio = days since last purchase divided by the average gap. A ratio above 1 means the customer is later than usual.

In [9]:
customer_rhythm["OverdueRatio"] = (
    customer_rhythm["DaysSinceLastPurchase"]
    / customer_rhythm["AverageDaysBetweenPurchases"]
)

customer_rhythm[[
    "Customer ID", "DaysSinceLastPurchase",
    "AverageDaysBetweenPurchases", "OverdueRatio"
]].head()


,Customer ID,DaysSinceLastPurchase,AverageDaysBetweenPurchases,OverdueRatio
0,12346,156,118.0,1.322034
1,12347,31,NaN,NaN
2,12348,65,NaN,NaN
3,12349,34,182.0,0.186813
4,12351,2,NaN,NaN


### Recent purchases

Invoices in the last 90 and 30 days, and purchase amount in the last 90 days. Use zero when there were no recent purchases.

In [10]:
recent_90d = history[
    history["InvoiceDate"] >= prediction_date - pd.Timedelta(days=90)
]

recent_30d = history[
    history["InvoiceDate"] >= prediction_date - pd.Timedelta(days=30)
]

recent_90 = recent_90d.groupby("Customer ID").agg(
    InvoiceCountLast90Days=("Invoice", "nunique"),
    PurchaseAmountLast90Days=("TransactionValue", "sum")
).reset_index()

recent_30 = (
    recent_30d.groupby("Customer ID")["Invoice"]
    .nunique()
    .reset_index(name="InvoiceCountLast30Days")
)

# Start from every customer, then add recent activity.
customer_recent = customer_features[["Customer ID"]].merge(
    recent_90, on="Customer ID", how="left"
)

customer_recent = customer_recent.merge(
    recent_30, on="Customer ID", how="left"
)

# No recent purchase means zero, not unknown.
customer_recent = customer_recent.fillna(0)

customer_recent.head()


,Customer ID,InvoiceCountLast90Days,PurchaseAmountLast90Days,InvoiceCountLast30Days
0,12346,0.0,0.00,0.0
1,12347,1.0,611.53,0.0
2,12348,1.0,221.16,0.0
3,12349,1.0,1152.62,0.0
4,12351,1.0,300.93,1.0


### Country

Country is the customer's country on their latest purchase. It stays as text and is encoded in the machine learning notebook.

In [11]:
customer_country = (
    history.sort_values("InvoiceDate")
    .groupby("Customer ID")["Country"]
    .last()
    .reset_index()
)

customer_country["Country"].value_counts().head(10)


Country
United Kingdom    3912
Germany             65
France              43
Netherlands         22
Spain               21
Portugal            16
Sweden              16
Belgium             15
Switzerland         14
Australia           13
Name: count, dtype: int64

# Combine the Features

Join all customer summaries by Customer ID and add the prediction date.

In [12]:
# Start from customer_features each time this cell runs.
features = customer_features.merge(
    customer_rhythm[[
        "Customer ID", "DaysSinceLastPurchase", "DaysSinceFirstPurchase",
        "AverageDaysBetweenPurchases", "OverdueRatio"
    ]],
    on="Customer ID", how="left"
)

features = features.merge(customer_recent, on="Customer ID", how="left")

features = features.merge(customer_country, on="Customer ID", how="left")

features["PredictionDate"] = prediction_date

features.shape


(4236, 14)

# Basic Checks

Each customer-date should appear once. The average gap and OverdueRatio can be missing for customers with one purchase day.

In [13]:
print("Repeated customer-dates:", features.duplicated(["Customer ID", "PredictionDate"]).sum())

features.isna().sum()


Repeated customer-dates: 0


Customer ID                           0
InvoiceCountLast365Days               0
PurchaseAmountLast365Days             0
DistinctProductCountLast365Days       0
AverageOrderValueLast365Days          0
DaysSinceLastPurchase                 0
DaysSinceFirstPurchase                0
AverageDaysBetweenPurchases        1491
OverdueRatio                       1491
InvoiceCountLast90Days                0
PurchaseAmountLast90Days              0
InvoiceCountLast30Days                0
Country                               0
PredictionDate                        0
dtype: int64

### Feature meanings

Unless another window is named, calculations use the previous 365 days.

| Feature | Meaning |
| --- | --- |
| InvoiceCountLast365Days | Number of distinct orders (Frequency). |
| PurchaseAmountLast365Days | Total amount spent (Monetary). |
| DistinctProductCountLast365Days | Number of different products bought. |
| AverageOrderValueLast365Days | Amount spent per order. |
| DaysSinceLastPurchase | Days since the customer's latest purchase (Recency). |
| DaysSinceFirstPurchase | Days since the first purchase in the history window; at most 365. |
| AverageDaysBetweenPurchases | Usual gap between purchase days; missing for one-day customers. |
| OverdueRatio | DaysSinceLastPurchase ÷ AverageDaysBetweenPurchases; above 1 means later than usual. |
| InvoiceCountLast90Days | Orders in the previous 90 days, including zero. |
| PurchaseAmountLast90Days | Amount spent in the previous 90 days, including zero. |
| InvoiceCountLast30Days | Orders in the previous 30 days, including zero. |
| Country | Country on the latest purchase; categorical, encoded later. |


# Create the Target

Churned = 1 if the customer makes no purchase in the 90 days starting at the prediction date. The full 90 days must be available in the data.

In [14]:
target_end = prediction_date + pd.Timedelta(days=90)

if target_end > df["InvoiceDate"].max():
    raise ValueError("Not enough future data for a full 90-day target.")

# Customers who bought anything during the next 90 days.
future_customers = df[
    (df["InvoiceDate"] >= prediction_date)
    & (df["InvoiceDate"] < target_end)
]["Customer ID"].unique()

model_data = features.copy()

# Churned = 1 when the customer is NOT among the future buyers.
model_data["Churned"] = (
    ~model_data["Customer ID"].isin(future_customers)
).astype(int)

model_data["Churned"].value_counts()


Churned
1    2831
0    1405
Name: count, dtype: int64

# Reuse the Same Steps

A function stores the calculations above under one name. Give it a prediction date; it returns that date's features and target.

In [15]:
def prepare_churn_data(prediction_date):
    prediction_date = pd.Timestamp(prediction_date).normalize()
    history_start = prediction_date - pd.Timedelta(days=365)
    target_end = prediction_date + pd.Timedelta(days=90)

    if history_start < df["InvoiceDate"].min().normalize():
        raise ValueError("Not enough data for a full 365-day history.")

    if target_end > df["InvoiceDate"].max():
        raise ValueError("Not enough future data for a full 90-day target.")

    # Purchase history.

    history = df[
        (df["InvoiceDate"] >= history_start)
        & (df["InvoiceDate"] < prediction_date)
    ].copy()

    history["TransactionValue"] = history["Quantity"] * history["Price"]

    # Frequency and monetary value.

    customer_features = history.groupby("Customer ID").agg(
        InvoiceCountLast365Days=("Invoice", "nunique"),
        PurchaseAmountLast365Days=("TransactionValue", "sum"),
        DistinctProductCountLast365Days=("StockCode", "nunique")
    ).reset_index()

    customer_features["AverageOrderValueLast365Days"] = (
        customer_features["PurchaseAmountLast365Days"]
        / customer_features["InvoiceCountLast365Days"]
    )

    # Recency and tenure.

    customer_dates = history.groupby("Customer ID").agg(
        FirstPurchaseDate=("InvoiceDate", "min"),
        LastPurchaseDate=("InvoiceDate", "max")
    ).reset_index()

    customer_dates["DaysSinceLastPurchase"] = (
        prediction_date - customer_dates["LastPurchaseDate"].dt.normalize()
    ).dt.days

    customer_dates["DaysSinceFirstPurchase"] = (
        prediction_date - customer_dates["FirstPurchaseDate"].dt.normalize()
    ).dt.days

    # Purchase rhythm and OverdueRatio.

    history["PurchaseDay"] = history["InvoiceDate"].dt.normalize()

    purchase_days = (
        history.groupby("Customer ID")["PurchaseDay"]
        .nunique()
        .reset_index(name="PurchaseDayCount")
    )

    customer_rhythm = customer_dates.merge(purchase_days, on="Customer ID", how="left")

    span_days = (
        customer_rhythm["LastPurchaseDate"].dt.normalize()
        - customer_rhythm["FirstPurchaseDate"].dt.normalize()
    ).dt.days

    gap_count = (customer_rhythm["PurchaseDayCount"] - 1).replace(0, float("nan"))

    customer_rhythm["AverageDaysBetweenPurchases"] = span_days / gap_count

    customer_rhythm["OverdueRatio"] = (
        customer_rhythm["DaysSinceLastPurchase"]
        / customer_rhythm["AverageDaysBetweenPurchases"]
    )

    # Recent purchases.

    recent_90d = history[
        history["InvoiceDate"] >= prediction_date - pd.Timedelta(days=90)
    ]

    recent_30d = history[
        history["InvoiceDate"] >= prediction_date - pd.Timedelta(days=30)
    ]

    recent_90 = recent_90d.groupby("Customer ID").agg(
        InvoiceCountLast90Days=("Invoice", "nunique"),
        PurchaseAmountLast90Days=("TransactionValue", "sum")
    ).reset_index()

    recent_30 = (
        recent_30d.groupby("Customer ID")["Invoice"]
        .nunique()
        .reset_index(name="InvoiceCountLast30Days")
    )

    customer_recent = customer_features[["Customer ID"]].merge(
        recent_90, on="Customer ID", how="left"
    )

    customer_recent = customer_recent.merge(
        recent_30, on="Customer ID", how="left"
    )

    customer_recent = customer_recent.fillna(0)

    # Country.

    customer_country = (
        history.sort_values("InvoiceDate")
        .groupby("Customer ID")["Country"]
        .last()
        .reset_index()
    )

    # Combine the features.

    features = customer_features.merge(
        customer_rhythm[[
            "Customer ID", "DaysSinceLastPurchase", "DaysSinceFirstPurchase",
            "AverageDaysBetweenPurchases", "OverdueRatio"
        ]],
        on="Customer ID", how="left"
    )

    features = features.merge(customer_recent, on="Customer ID", how="left")

    features = features.merge(customer_country, on="Customer ID", how="left")

    features["PredictionDate"] = prediction_date

    # Create the target.

    future_customers = df[
        (df["InvoiceDate"] >= prediction_date)
        & (df["InvoiceDate"] < target_end)
    ]["Customer ID"].unique()

    features["Churned"] = (
        ~features["Customer ID"].isin(future_customers)
    ).astype(int)

    return features


### Repeat December

Run the function for 1 December 2010. It should reproduce the table above.

In [16]:
december_check = prepare_churn_data("2010-12-01")

print(december_check.shape)

december_check.equals(model_data)


(4236, 15)


True

### Prepare all months

Monthly prediction dates from December 2010 to September 2011. September is the last date with a full 90-day outcome, because the data ends on 9 December 2011.

In [17]:
prediction_dates = pd.date_range("2010-12-01", "2011-09-01", freq="MS")

monthly_tables = []

for date in prediction_dates:
    monthly_tables.append(prepare_churn_data(date))

all_months_data = pd.concat(monthly_tables, ignore_index=True)

all_months_data.groupby("PredictionDate").agg(
    Customers=("Churned", "size"),
    ChurnRate=("Churned", "mean")
).round(3)


,Customers,ChurnRate
PredictionDate,,
2010-12-01,4236,0.668
2011-01-01,4202,0.670
2011-02-01,4218,0.654
2011-03-01,4258,0.631
2011-04-01,4316,0.622
2011-05-01,4317,0.613
2011-06-01,4324,0.634
2011-07-01,4313,0.614
2011-08-01,4325,0.569


# Final Table

Each row has 12 features, Customer ID, PredictionDate and Churned. The train-test split is created in the machine learning notebook.

In [18]:
final_data = all_months_data

print(final_data.shape)

final_data.head()


(42832, 15)


,Customer ID,InvoiceCountLast365Days,PurchaseAmountLast365Days,DistinctProductCountLast365Days,AverageOrderValueLast365Days,DaysSinceLastPurchase,DaysSinceFirstPurchase,AverageDaysBetweenPurchases,OverdueRatio,InvoiceCountLast90Days,PurchaseAmountLast90Days,InvoiceCountLast30Days,Country,PredictionDate,Churned
0,12346,2,169.36,24,84.68,156,274,118.0,1.322034,0.0,0.00,0.0,United Kingdom,2010-12-01,1
1,12347,1,611.53,40,611.53,31,31,NaN,NaN,1.0,611.53,0.0,Iceland,2010-12-01,0
2,12348,1,221.16,19,221.16,65,65,NaN,NaN,1.0,221.16,0.0,Finland,2010-12-01,0
3,12349,2,2221.14,89,1110.57,34,216,182.0,0.186813,1.0,1152.62,0.0,Italy,2010-12-01,1
4,12351,1,300.93,21,300.93,2,2,NaN,NaN,1.0,300.93,1.0,Unspecified,2010-12-01,1


### Basic checks

In [19]:
print("Repeated customer-dates:", final_data.duplicated(["Customer ID", "PredictionDate"]).sum())

print(final_data["Churned"].value_counts())

final_data.isna().sum()


Repeated customer-dates: 0
Churned
1    26455
0    16377
Name: count, dtype: int64


Customer ID                            0
InvoiceCountLast365Days                0
PurchaseAmountLast365Days              0
DistinctProductCountLast365Days        0
AverageOrderValueLast365Days           0
DaysSinceLastPurchase                  0
DaysSinceFirstPurchase                 0
AverageDaysBetweenPurchases        15848
OverdueRatio                       15848
InvoiceCountLast90Days                 0
PurchaseAmountLast90Days               0
InvoiceCountLast30Days                 0
Country                                0
PredictionDate                         0
Churned                                0
dtype: int64

### Save the final table

In [20]:
final_data.to_csv("data/churn_data.csv", index=False)

# Exploratory Data Analysis

**Question: which customers stop buying?**

We explore only the **training months (December 2010 - May 2011)**. The test months (August - September 2011) stay unseen until the final evaluation, so nothing we learn here leaks from the test.

In [21]:
import plotly.express as px

eda = final_data[final_data["PredictionDate"] <= "2011-05-01"].copy()

eda.shape

(25547, 15)

### 1. Target: how many customers churn? (pie chart)

In [22]:
eda["Churned"].value_counts(normalize=True).round(3)

Churned
1    0.643
0    0.357
Name: proportion, dtype: float64

In [23]:
px.pie(eda, names="Churned", title="Churned (1) vs bought again (0)")

About **64%** of customer snapshots churn and **36%** buy again. Churn is the **majority** class, so the imbalance is moderate, not extreme like fraud.

### 2. Recency: days since the last purchase (histogram)

In [24]:
eda.groupby("Churned")["DaysSinceLastPurchase"].median()

Churned
0     47.0
1    117.0
Name: DaysSinceLastPurchase, dtype: float64

In [25]:
px.histogram(eda, x="DaysSinceLastPurchase", color="Churned", barmode="overlay",
             title="Days since last purchase, by churn")

Customers who churn have been away much longer: a median of **117 days**, against **47 days** for customers who buy again.

### 3. Frequency: orders in the past year (bar chart)

In [26]:
eda["OrderBand"] = pd.cut(eda["InvoiceCountLast365Days"], [0, 1, 3, 10, float("inf")],
                          labels=["1 order", "2-3 orders", "4-10 orders", "11+ orders"])

order_churn = eda.groupby("OrderBand", observed=True)["Churned"].mean().reset_index()

order_churn

,OrderBand,Churned
0,1 order,0.847197
1,2-3 orders,0.715979
2,4-10 orders,0.436011
3,11+ orders,0.116086


In [27]:
px.bar(order_churn, x="OrderBand", y="Churned", title="Churn rate by number of orders")

The more often a customer orders, the less likely they are to leave: **85%** of one-order customers churn, but only **12%** of customers with 11 or more orders.

### 4. Monetary value: spending in the past year (box plot)

In [28]:
eda.groupby("Churned")["PurchaseAmountLast365Days"].median().round(0)

Churned
0    1496.0
1     458.0
Name: PurchaseAmountLast365Days, dtype: float64

In [29]:
px.box(eda, x="Churned", y="PurchaseAmountLast365Days", log_y=True,
       title="Spending in the past year, by churn (log scale)")

Customers who stay spend more: a median of about **£1,496**, against **£458** for customers who churn. The log scale makes the long tail of big wholesale buyers readable.

### 5. OverdueRatio: late compared with their own habit (bar chart)

In [30]:
eda["OverdueBand"] = pd.cut(eda["OverdueRatio"], [0, 0.5, 1, 2, float("inf")],
                            labels=["Below 0.5", "0.5-1", "1-2", "Above 2"], include_lowest=True)

eda["OverdueBand"] = eda["OverdueBand"].astype("object").fillna("One purchase day")

overdue_churn = eda.groupby("OverdueBand")["Churned"].mean().reset_index()

overdue_churn

,OverdueBand,Churned
0,0.5-1,0.463231
1,1-2,0.492698
2,Above 2,0.657534
3,Below 0.5,0.477409
4,One purchase day,0.847583


In [31]:
px.bar(overdue_churn, x="OverdueBand", y="Churned", title="Churn rate by OverdueRatio",
       category_orders={"OverdueBand": ["Below 0.5", "0.5-1", "1-2", "Above 2", "One purchase day"]})

Customers more than twice their usual gap behind churn **66%** of the time, against about **47%** for customers who are not yet due. Customers with only one purchase day, who have no habit yet, churn the most (**85%**).

### 6. Season: churn rate by prediction month (line chart)

In [32]:
month_churn = eda.groupby("PredictionDate")["Churned"].mean().reset_index()

px.line(month_churn, x="PredictionDate", y="Churned", markers=True, title="Churn rate by prediction month")

Churn falls from about **67%** in December-January to **61%** in May, as customers start buying for the next season. The test months (August-September) are just before Christmas, so even fewer customers are expected to leave there.

### 7. Country (bar chart)

In [33]:
eda["CountryGroup"] = eda["Country"].where(eda["Country"].isin(["United Kingdom", "Germany", "France"]), "Other")

country_churn = eda.groupby("CountryGroup")["Churned"].agg(["mean", "size"]).reset_index()

country_churn

,CountryGroup,mean,size
0,France,0.445902,305
1,Germany,0.498801,417
2,Other,0.661940,1340
3,United Kingdom,0.646966,23485


In [34]:
px.bar(country_churn, x="CountryGroup", y="mean", title="Churn rate by country")

German (**50%**) and French (**45%**) customers churn less than UK customers (**65%**), but 92% of rows are from the UK and the other countries are small.

### 8. Frequency and spending together (scatter plot)

In [35]:
px.scatter(eda.sample(3000, random_state=42), x="InvoiceCountLast365Days", y="PurchaseAmountLast365Days",
           color="Churned", log_x=True, log_y=True, opacity=0.6,
           title="Orders vs spending in the past year (sample of 3,000)")

Churners (1) sit in the bottom-left corner: few orders and low spending. Frequent, high-spending customers almost always stay.

### 9. Correlations between numerical features (heatmap)

In [36]:
num_cols = eda.select_dtypes("number").columns

px.imshow(eda[num_cols].corr().round(2), text_auto=True, aspect="auto",
          color_continuous_scale="RdBu_r", title="Correlation heatmap")

- **Churned** correlates most with DaysSinceLastPurchase (+0.31) and InvoiceCountLast365Days (-0.31).
- Some features repeat each other, for example InvoiceCountLast30Days and InvoiceCountLast90Days (0.78). This is checked again in feature selection.

### EDA conclusions

1. Churn is the majority class (64%); the imbalance is moderate.
2. **Recency, frequency and spending** separate churners from customers who stay the most.
3. **OverdueRatio** adds a habit-based view: being late compared with your own rhythm raises churn.
4. Churn changes with the **season**, which matters because the test months are just before Christmas.
5. **Country** has some differences, but most customers are from the UK.

Continue with `3_Machine_Learning.ipynb`.